# OpenCode Web — User Manager

Programmatic alternative to the admin console at `/__oc_admin`. Run the cells in order:
configure → client → bulk create → bulk delete → clear chat history → status.

Endpoints used (implemented by the router, see `templates/configmap-router.yaml`):

- `GET  /__oc_admin/api/users` — list users
- `POST /__oc_admin/api/create` — `{username, password}`
- `POST /__oc_admin/api/delete` — `{username}` (removes the account **and its pods/PVCs**)
- `POST /__oc_admin/api/reset-password` — `{username, password}`
- `POST /__oc_admin/api/rename` — `{username, newUsername}`
- `POST /__oc_admin/api/sessions/clear` — `{username}` or `{all: true}` (deletes opencode chat
  history from the user's state PVC; workspace files, configs and model keys are kept)

Authentication is HTTP Basic using the `admin.username` / `admin.password` from your
values file. Username rules: 3–64 chars, lowercase letters, digits, `.`, `_`, `-`.
Password rule: at least 6 characters.

## 1. Configuration

Set the values below, or export `OPENCODE_BASE_URL`, `OPENCODE_ADMIN_USER` and
`OPENCODE_ADMIN_PASSWORD` in the environment before starting Jupyter.

In [ ]:
import os
import re
import secrets
import string

import pandas as pd
import requests

BASE_URL = os.environ.get('OPENCODE_BASE_URL', 'http://opencode-web-helm-svc.opencode-web-helm.svc:8080').rstrip('/')
ADMIN_USER = os.environ.get('OPENCODE_ADMIN_USER', 'admin-user')
ADMIN_PASSWORD = os.environ.get('OPENCODE_ADMIN_PASSWORD', 'admin-password')
USERNAME_RE = re.compile(r'^[a-z0-9._-]{3,64}$')

class AdminApiError(RuntimeError):
    def __init__(self, status, message):
        super().__init__(f'HTTP {status}: {message}')
        self.status = status
        self.message = message


def _admin_request(method, path, json_body=None, timeout=120):
    response = requests.request(method, BASE_URL + path, json=json_body, auth=(ADMIN_USER, ADMIN_PASSWORD), timeout=timeout)
    if response.status_code >= 400:
        try:
            message = response.json().get('error', response.text)
        except ValueError:
            message = response.text
        raise AdminApiError(response.status_code, message)
    if not response.content:
        return {}
    try:
        return response.json()
    except ValueError as error:
        content_type = response.headers.get('content-type', '')
        raise AdminApiError(response.status_code, (
            f'response is not JSON (status {response.status_code}, content-type: {content_type!r}, final url: {response.url}). '
            'This usually means the platform SSO (oauth2-proxy) intercepted the request at the ingress gateway. '
            'Run this notebook from a pod inside the cluster and set '
            "BASE_URL='http://<release-name>-svc.<namespace>.svc:8080' (the router service has no SSO), "
            'or authenticate with platform SSO first. '
            f'Body starts with: {response.text[:200]!r}'
        )) from error

print('config loaded, target:', BASE_URL)

## 2. API client

In [ ]:
def list_users():
    return _admin_request('GET', '/__oc_admin/api/users').get('users', [])


def create_user(username, password=None):
    username = username.strip().lower()
    if not USERNAME_RE.match(username):
        raise ValueError('username must be 3-64 chars: lowercase letters, digits, . _ -')
    if password is None:
        alphabet = string.ascii_letters + string.digits
        password = ''.join(secrets.choice(alphabet) for _ in range(12))
    if len(password) < 6:
        raise ValueError('password must be at least 6 characters')
    _admin_request('POST', '/__oc_admin/api/create', {'username': username, 'password': password})
    return {'username': username, 'password': password}


def delete_user(username):
    username = username.strip().lower()
    _admin_request('POST', '/__oc_admin/api/delete', {'username': username})
    return {'username': username, 'deleted': True}


def reset_password(username, password):
    username = username.strip().lower()
    if len(password) < 6:
        raise ValueError('password must be at least 6 characters')
    _admin_request('POST', '/__oc_admin/api/reset-password', {'username': username, 'password': password})
    return {'username': username, 'password': password}


def rename_user(username, new_username):
    username = username.strip().lower()
    new_username = new_username.strip().lower()
    if not USERNAME_RE.match(new_username):
        raise ValueError('new username must be 3-64 chars: lowercase letters, digits, . _ -')
    _admin_request('POST', '/__oc_admin/api/rename', {'username': username, 'newUsername': new_username})
    return {'old': username, 'new': new_username}

print('admin api client ready')

## 3. Bulk create

`create_users('devday', 20)` creates `devday1` … `devday20` with a distinct auto-generated
password per user, and prints a credentials table to hand out. Already-existing users are
skipped. Use `password='workshop123'` to give everyone the same password instead.

In [ ]:
def create_users(prefix, count, password=None, start=1):
    prefix = prefix.strip().lower()
    if not USERNAME_RE.match(f'{prefix}{start}'):
        raise ValueError('prefix + index must form a valid username: lowercase letters, digits, . _ - (3-64 chars)')
    existing = {user['username'] for user in list_users()}
    results = []
    for i in range(start, start + count):
        username = f'{prefix}{i}'
        if username in existing:
            results.append({'username': username, 'password': '', 'status': 'skipped (already exists)'})
            continue
        try:
            creds = create_user(username, password)
            results.append({'username': username, 'password': creds['password'], 'status': 'created'})
        except (AdminApiError, ValueError) as error:
            results.append({'username': username, 'password': '', 'status': f'failed: {error}'})
    frame = pd.DataFrame(results)
    print(f'create_users(prefix={prefix!r}, count={count}, start={start})')
    print(frame.to_string(index=False) if len(frame) else 'nothing to create')
    return frame

create_users('devday', 20)

## 4. Delete users

`delete_users(['devday1', 'devday2'])` deletes the listed users (a single string works too).
`delete_bulk_users('devday')` deletes every user whose username starts with the prefix.
`change_password(['devday1', 'devday2'])` resets passwords: auto-generates a distinct
password per user unless you pass a shared `password='workshop123'`, and prints the new
credentials. `change_bulk_password('devday')` does the same for every user whose username
starts with the prefix. All default to safe behavior; deletion additionally requires
`dry_run=False`. Deletion removes each account and all of its pods/PVCs — irreversible.

In [ ]:
def delete_bulk_users(prefix, dry_run=True):
    prefix = prefix.strip().lower()
    targets = [user['username'] for user in list_users() if isinstance(user['username'], str) and user['username'].startswith(prefix)]
    if not targets:
        print(f'no users found with prefix {prefix!r}')
        return pd.DataFrame()
    if dry_run:
        print(f'DRY RUN — {len(targets)} user(s) would be deleted (including pods and PVCs):')
        for username in targets:
            print(' -', username)
        print('re-run with dry_run=False to actually delete')
        return pd.DataFrame({'username': targets, 'status': 'would delete'})
    results = []
    for username in targets:
        try:
            delete_user(username)
            results.append({'username': username, 'status': 'deleted'})
        except AdminApiError as error:
            results.append({'username': username, 'status': f'failed: {error}'})
    frame = pd.DataFrame(results)
    print(frame.to_string(index=False))
    return frame


def delete_users(usernames, dry_run=True):
    if isinstance(usernames, str):
        usernames = [usernames]
    usernames = [username.strip().lower() for username in usernames]
    users = {user['username']: user for user in list_users()}
    found = [username for username in usernames if username in users]
    missing = [username for username in usernames if username not in users]
    if not found:
        print('none of the requested users exist', ('(' + ', '.join(missing) + ')') if missing else '')
        return pd.DataFrame()
    if dry_run:
        print(f'DRY RUN — {len(found)} user(s) would be deleted (including pods and PVCs):')
        for username in found:
            print(' -', username, f"(ready: {users[username].get('ready')})")
        if missing:
            print('not found (skipped):', ', '.join(missing))
        print('re-run with dry_run=False to actually delete')
        return pd.DataFrame([{'username': username, 'status': 'would delete'} for username in found])
    results = []
    for username in found:
        try:
            delete_user(username)
            results.append({'username': username, 'status': 'deleted'})
        except AdminApiError as error:
            results.append({'username': username, 'status': f'failed: {error}'})
    for username in missing:
        results.append({'username': username, 'status': 'not found'})
    frame = pd.DataFrame(results)
    print(frame.to_string(index=False))
    return frame


def change_password(usernames, password=None):
    if isinstance(usernames, str):
        usernames = [usernames]
    if password is not None and len(password) < 6:
        raise ValueError('password must be at least 6 characters')
    usernames = [username.strip().lower() for username in usernames]
    users = {user['username']: user for user in list_users()}
    results = []
    for username in usernames:
        if username not in users:
            results.append({'username': username, 'password': '', 'status': 'not found'})
            continue
        new_password = password or ''.join(secrets.choice(string.ascii_letters + string.digits) for _ in range(12))
        try:
            reset_password(username, new_password)
            results.append({'username': username, 'password': new_password, 'status': 'updated'})
        except AdminApiError as error:
            results.append({'username': username, 'password': '', 'status': f'failed: {error}'})
    frame = pd.DataFrame(results)
    print(frame.to_string(index=False))
    return frame


def change_bulk_password(prefix, password=None):
    prefix = prefix.strip().lower()
    targets = [user['username'] for user in list_users() if isinstance(user['username'], str) and user['username'].startswith(prefix)]
    if not targets:
        print(f'no users found with prefix {prefix!r}')
        return pd.DataFrame()
    return change_password(targets, password=password)

delete_users(['devday1', 'devday2'], dry_run=True)

## 5. Clear chat history (refresh conversations)

Deletes each user's opencode conversation database (`opencode.db` under
`/var/opencode/data/opencode` on the user's state PVC; legacy `storage/` subtrees
included), so every user starts with an empty chat history. The router runs a
short-lived cleanup Job per user and then restarts the user's pod so the running
server picks up the fresh, empty database — requires the updated chart to be deployed.

- `clear_all_sessions()` — every registry user (the one-line "refresh each user")
- `clear_bulk_sessions('devday')` — every user whose username starts with the prefix
- `clear_user_sessions(['devday1', 'devday2'])` — an explicit list
- Environments without a registered username (shown as "unknown" in the admin console)
  are skipped — remove those from the admin console instead

Deleted: the conversation database (`opencode.db`, sessions/messages). Kept: workspace
files, `opencode.json` / agents / skills, `auth.json` (model API keys). Users stay
logged in; each cleared user's pod restarts briefly (seconds). Irreversible.

In [ ]:
def _clear_sessions_rows(payload):
    return [
        {'username': entry.get('username'), 'slug': entry.get('slug'), 'status': entry.get('status')}
        for entry in payload.get('results', [])
    ]


def clear_user_sessions(usernames, dry_run=True):
    if isinstance(usernames, str):
        usernames = [usernames]
    usernames = [username.strip().lower() for username in usernames if isinstance(username, str)]
    users = {user['username'] for user in list_users()}
    targets = [username for username in usernames if username in users]
    missing = [username for username in usernames if username not in users]
    if not targets:
        print('none of the requested users exist', ('(' + ', '.join(missing) + ')') if missing else '')
        return pd.DataFrame()
    if dry_run:
        print(f'DRY RUN — chat history would be cleared for {len(targets)} user(s):')
        print('(workspace files, configs and model keys are kept; users stay logged in)')
        for username in targets:
            print(' -', username)
        if missing:
            print('not found (skipped):', ', '.join(missing))
        print('re-run with dry_run=False to actually clear')
        return pd.DataFrame([{'username': username, 'status': 'would clear'} for username in targets])
    results = []
    for username in targets:
        try:
            payload = _admin_request('POST', '/__oc_admin/api/sessions/clear', {'username': username}, timeout=300)
            results.extend(_clear_sessions_rows(payload))
        except AdminApiError as error:
            results.append({'username': username, 'slug': '', 'status': f'failed: {error}'})
    for username in missing:
        results.append({'username': username, 'slug': '', 'status': 'not found'})
    frame = pd.DataFrame(results)
    print(frame.to_string(index=False))
    return frame


def clear_bulk_sessions(prefix, dry_run=True):
    prefix = prefix.strip().lower()
    targets = [user['username'] for user in list_users() if isinstance(user['username'], str) and user['username'].startswith(prefix)]
    if not targets:
        print(f'no users found with prefix {prefix!r}')
        return pd.DataFrame()
    return clear_user_sessions(targets, dry_run=dry_run)


def clear_all_sessions(dry_run=True):
    if dry_run:
        return clear_user_sessions([user['username'] for user in list_users() if isinstance(user['username'], str)], dry_run=True)
    payload = _admin_request('POST', '/__oc_admin/api/sessions/clear', {'all': True}, timeout=600)
    frame = pd.DataFrame(_clear_sessions_rows(payload))
    print(f'clear_all_sessions() — {len(frame)} user(s) processed')
    print(frame.to_string(index=False) if len(frame) else 'no users in registry')
    return frame

clear_all_sessions(dry_run=True)

## 6. Current users and provisioning status

In [ ]:
def users_frame():
    return pd.DataFrame(list_users())

users_frame()

## Troubleshooting: `response is not JSON` / JSONDecodeError

The public endpoint (`https://opencode.<domain>`) is guarded by the platform SSO
(oauth2-proxy `AuthorizationPolicy` at the ingress gateway). Without an SSO session
you receive an HTML login page instead of the API response.

Run the diagnostic below to see what actually came back. If it shows a login page,
either:

1. **Run the notebook from a pod inside the cluster** (any user pod works) and set
   `BASE_URL` to the router service directly — it has no SSO in front of it:
   `BASE_URL = 'http://<release-name>-svc.<namespace>.svc:8080'`. Find the name with
   `kubectl get svc -A | grep opencode` (or from a pod: `nslookup` the guessed name).
2. **Open the domain in your browser first** (SSO login), then reuse — not practical
   for scripts; prefer option 1.

In [ ]:
r = requests.get(BASE_URL + '/__oc_admin/api/users', auth=(ADMIN_USER, ADMIN_PASSWORD), timeout=30)
print('final url:', r.url)
print('status:', r.status_code)
print('content-type:', r.headers.get('content-type'))
print(r.text[:400])